# v0.3 S1-HP — HotpotQA retrieval (E1): sub-corpus, 7 retrievers
**How to run:** Settings → Accelerator **GPU T4 x2**, Internet **On**. Then **Save Version → Save & Run All (Commit)**. It runs unattended (about 3–4 h). If a step fails, the notebook stops with `FAILED` — send me the log. When it finishes, open the version's **Output** tab, download `v03_s1hp_results.zip` and send it to me.

In [ ]:
# Cell 0 — helper: run a shell command, stream its output, STOP the notebook if it fails
import os, subprocess
def sh(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {cmd}")

In [ ]:
# Cell 0b — run two commands at once (one per GPU); STOP if either fails
import threading
def sh_par(*cmds):
    errs = []
    def run(c):
        try:
            sh(c)
        except Exception as e:
            errs.append(str(e))
    ts = [threading.Thread(target=run, args=(c,)) for c in cmds]
    [t.start() for t in ts]; [t.join() for t in ts]
    if errs:
        raise RuntimeError("; ".join(errs))

In [ ]:
# Cell 1 — hardware, code frozen at 8c21832 (branch v03), pinned environment, check, tests
sh("nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv || true")
sh("df -h / /tmp /kaggle/working && free -g && nproc")
os.chdir("/kaggle/working")
try:                                   # Hugging Face token from Kaggle Secrets (Add-ons -> Secrets -> HF_TOKEN)
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); print("HF_TOKEN loaded")
except Exception as e:
    print("HF_TOKEN not available (public downloads still work):", type(e).__name__)
# the GitHub repo is private: the code comes from a git bundle uploaded as a private Kaggle dataset
import glob
bundles = glob.glob("/kaggle/input/**/rag_v03*.bundle", recursive=True)
assert len(bundles) == 1, f"attach the code dataset with exactly one rag_v03*.bundle, found {bundles}"
sh(f"git clone -q {bundles[0]} rag")
os.chdir("/kaggle/working/rag")
sh("git checkout -q 8c21832791245f29f087179cadd41b1d427ef317 && git log --oneline -1")
sh("test \"$(git rev-parse HEAD)\" = 8c21832791245f29f087179cadd41b1d427ef317 || (echo WRONG CODE COMMIT; exit 1)")
sh("pip install -q -r requirements-retrieval-pinned.txt")
sh("python -m src.check_env --lock requirements-retrieval.lock")
sh("python -m src.env_report --session v03_S1hp --out results/v03_hp/raw/env_v03_S1hp.json")
sh("python -m pytest -q -p no:cacheprovider 2>&1 | tail -3")

In [ ]:
# Cell 2 — E1 data: HotpotQA (BEIR corpus 5.2M passages, test qrels, answers), pinned at first download; join by id (gate >= 95%)
sh("python -m src.data.download --config configs/v03_hp_qwen.yaml")
sh("python -m src.data.join_hotpot --config configs/v03_hp_qwen.yaml")

In [ ]:
# Cell 3 — sub-corpus: BM25 over the FULL corpus for all judged queries, then 300,000 passages (gold + BM25 negatives + random)
sh("python -m src.retrieval.bm25 --config configs/v03_hp_qwen.yaml --corpus full --queries judged")
sh("python -m src.data.subcorpus --config configs/v03_hp_qwen.yaml --size 300000 --population judged --name main")

In [ ]:
# Cell 4 — R0: BM25 on the sub-corpus, all judged queries
sh("python -m src.retrieval.bm25 --config configs/v03_hp_qwen.yaml --corpus main --queries judged")

In [ ]:
# Cell 5 — encode the 300k sub-corpus + all judged queries: GPU 0 = R1,R2,R3,R5, GPU 1 = R4,R6 (in parallel, ~1.7 h)
sh_par("CUDA_VISIBLE_DEVICES=0 python -m src.retrieval.encode --config configs/v03_hp_qwen.yaml --models R1,R2,R3,R5 --corpus main",
       "CUDA_VISIBLE_DEVICES=1 python -m src.retrieval.encode --config configs/v03_hp_qwen.yaml --models R4,R6 --corpus main")

In [ ]:
# Cell 6 — exact search (CPU), all judged queries
sh("python -m src.retrieval.search --config configs/v03_hp_qwen.yaml --model R1 --corpus main --queries judged")
sh("python -m src.retrieval.search --config configs/v03_hp_qwen.yaml --model R2 --corpus main --queries judged")
sh("python -m src.retrieval.search --config configs/v03_hp_qwen.yaml --model R3 --corpus main --queries judged")
sh("python -m src.retrieval.search --config configs/v03_hp_qwen.yaml --model R5 --corpus main --queries judged")
sh("python -m src.retrieval.search --config configs/v03_hp_qwen.yaml --model R4 --corpus main --queries judged")
sh("python -m src.retrieval.search --config configs/v03_hp_qwen.yaml --model R6 --corpus main --queries judged")

In [ ]:
# Last cell — package (becomes this version's saved output)
sh("ls results/v03_hp/raw/runs | head -40")
sh("zip -qr /kaggle/working/v03_s1hp_results.zip results/v03_hp data/processed/hp_* configs/resolved_revisions.json logs requirements-*.lock data/raw/hp_qrels/test.tsv data/raw/hp_corpus/queries")
sh("ls -la /kaggle/working/v03_s1hp_results.zip")
sh("rm -rf /kaggle/working/rag/data/raw /kaggle/working/rag/data/embeddings_hp /tmp/models")